# Cell 1 - Load data + Setup


In [0]:
# Cell 1 - Load data + Setup
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Load labeled data
df = pd.read_parquet("/Volumes/workspace/default/retail_data/customer_labeled.parquet")

# Check MLflow tracking URI (Databricks tự setup sẵn)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Shape:", df.shape)
print("\nSegment distribution:")
print(df["segment"].value_counts())

MLflow tracking URI: databricks
Shape: (5878, 20)

Segment distribution:
segment
Potential    1682
Champions    1681
Loyal        1424
At Risk      1091
Name: count, dtype: int64


# CELL 2 - Train Model 

In [0]:
%pip install xgboost

In [0]:
# Cell 2 - Train 3 models + MLflow tracking
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import mlflow
import mlflow.sklearn

# Feature columns
feature_cols = [
    "total_spent", "avg_unit_price", "num_invoices", "num_line_items",
    "num_unique_products", "num_countries", "recency_days", "active_days",
    "purchase_frequency", "avg_basket_value",
    "log_total_spent", "log_num_invoices", "log_num_unique_products", "log_avg_basket_value"
]

X = df[feature_cols]
y = df["segment"]

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Classes: {le.classes_}")

# Define 3 models
models = {
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=200, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=42, eval_metric="mlogloss", verbosity=0),
}

# Train + log to MLflow
experiment_name = "/Users/lethanhcong.hcmus@gmail.com/retail-mlops-customer-segmentation"
mlflow.set_experiment(experiment_name)
results = {}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        # Train
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        # Log to MLflow
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1_macro)
        mlflow.log_metric("f1_weighted", f1_weighted)
        mlflow.sklearn.log_model(model, artifact_path="model")

        results[model_name] = {
            "accuracy": acc,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted
        }

        print(f"\n{'='*40}")
        print(f"Model: {model_name}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"F1 Macro  : {f1_macro:.4f}")
        print(f"F1 Weighted: {f1_weighted:.4f}")

# Summary
print("\n=== SUMMARY ===")
results_df = pd.DataFrame(results).T.sort_values("f1_macro", ascending=False)
print(results_df.round(4))

Train: (4702, 14) | Test: (1176, 14)
Classes: ['At Risk' 'Champions' 'Loyal' 'Potential']


2026/04/10 08:42:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de5f4f38-ef8f.cloud.databricks.com/ml/experiments/2376441762122699/models/m-359adb56ce794694b02966ce624cfb50?o=7474649648166776
2026/04/10 08:42:41 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



Model: HistGradientBoosting
Accuracy  : 0.9983
F1 Macro  : 0.9982
F1 Weighted: 0.9983


2026/04/10 08:42:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de5f4f38-ef8f.cloud.databricks.com/ml/experiments/2376441762122699/models/m-ca89d150d7924809b668c1a675039312?o=7474649648166776
2026/04/10 08:42:47 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



Model: RandomForest
Accuracy  : 0.9957
F1 Macro  : 0.9959
F1 Weighted: 0.9957


2026/04/10 08:42:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de5f4f38-ef8f.cloud.databricks.com/ml/experiments/2376441762122699/models/m-b04e728e7fc446f1acf973f877c048f3?o=7474649648166776
2026/04/10 08:42:52 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



Model: XGBoost
Accuracy  : 0.9983
F1 Macro  : 0.9984
F1 Weighted: 0.9983

=== SUMMARY ===
                      accuracy  f1_macro  f1_weighted
XGBoost                 0.9983    0.9984       0.9983
HistGradientBoosting    0.9983    0.9982       0.9983
RandomForest            0.9957    0.9959       0.9957


##  Model Training Results

### Metrics Summary:
| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| **XGBoost** | 0.9983 | **0.9984** | 0.9983 |
| HistGradientBoosting | 0.9983 | 0.9982 | 0.9983 |
| RandomForest | 0.9957 | 0.9959 | 0.9957 |

### Winner: XGBoost 
XGBoost đạt F1 Macro cao nhất (0.9984) - chênh lệch nhỏ so với
HistGradientBoosting nhưng đây là tiebreaker hợp lý.

### Tại sao accuracy cao đến vậy (~99%)?
Labels được tạo bằng rule-based RFM từ chính các features này
→ model học lại được rules đó rất dễ.
Đây là expected behavior, không phải data leakage.
Trong production, labels sẽ được validate bởi business team
trước khi dùng làm ground truth.

### MLflow Tracking:
- 3 runs được log đầy đủ: params, metrics, model artifacts
- Experiment: `retail-mlops-customer-segmentation`

# Cell 3 - Register best model


In [0]:


# Lấy run_id của XGBoost 
runs = mlflow.search_runs(
    experiment_names=["/Users/lethanhcong.hcmus@gmail.com/retail-mlops-customer-segmentation"],
    order_by=["metrics.f1_macro DESC"]
)

best_run_id = runs.iloc[0]["run_id"]
best_model_name = runs.iloc[0]["params.model_type"]
print(f"Best model: {best_model_name}")
print(f"Run ID: {best_run_id}")
print(f"F1 Macro: {runs.iloc[0]['metrics.f1_macro']:.4f}")

Best model: XGBoost
Run ID: a7a304958b324d6ca32be9855da9a04a
F1 Macro: 0.9984


# Cell 4 — Register model


In [0]:
#model_uri = f"runs:/{best_run_id}/model"
#model_name = "retail-customer-segmentation"

#registered = mlflow.register_model(
#    model_uri=model_uri,
#    name=model_name
#)

#print(f"Model registered!")
#print(f"Name   : {registered.name}")
#print(f"Version: {registered.version}")
#print(f"Status : {registered.status}")

# NOTE: Bỏ qua cell này - Unity Catalog yêu cầu model signature
# đã được fix và register đúng ở Cell 5

# # Cell 5 — Retrain XGBoos

In [0]:
from mlflow.models.signature import infer_signature
import mlflow.xgboost

xgb_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric="mlogloss", verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

# Signature + input example
signature = infer_signature(X_train, y_pred)
input_example = X_train.iloc[:5]

with mlflow.start_run(run_name="XGBoost_final"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))

    # Dùng mlflow.xgboost thay vì mlflow.sklearn
    mlflow.xgboost.log_model(
        xgb_model,
        name="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="workspace.default.retail-customer-segmentation"
    )

    print("Done!")
    print(f"F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-de5f4f38-ef8f.cloud.databricks.com/ml/experiments/2376441762122699/models/m-20de57c16be64e66b9c3731a5ad322b2?o=7474649648166776
Registered model 'workspace.default.retail-customer-segmentation' already ex

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.retail-customer-segmentation': https://dbc-de5f4f38-ef8f.cloud.databricks.com/explore/data/models/workspace/default/retail-customer-segmentation/version/1?o=7474649648166776


Done!
F1 Macro: 0.9984


## Model Registry

XGBoost model đã được register vào **Databricks Unity Catalog**:
- Model name: `workspace.default.retail-customer-segmentation`
- Version: 1
- F1 Macro: 0.9984

### MLflow tracking đã log đầy đủ:
- Parameters: model_type, n_features, train_size
- Metrics: accuracy, f1_macro, f1_weighted
- Artifacts: model binary + signature + input_example

### Model Signature:
Input schema được infer từ training data (14 features).
Output schema là integer encoded labels (0-3 → 4 segments).
Signature đảm bảo model có thể được serve và validate
input/output đúng chuẩn trong production.